# Text Reconstruction with Sequence Autoencoders


**Learning Objectives**:
- Understand sequence-to-sequence (seq2seq) autoencoders
- Implement encoder-decoder architectures for text
- Learn about teacher forcing and training strategies
- Visualize and explore learned latent representations
- Experiment with latent space interpolation
- Explore applications: compression, paraphrasing, generation

**What are Text Autoencoders?**

Autoencoders compress data into a lower-dimensional latent space and then reconstruct it. For text:
- **Encoder**: Compresses variable-length sentences into fixed-size latent vectors
- **Latent Space**: Compact representation capturing sentence semantics
- **Decoder**: Reconstructs the original sentence from the latent vector

**Why are they useful?**
- **Unsupervised learning**: Learn representations without labeled data
- **Dimensionality reduction**: Compress sentences to fixed-size vectors
- **Generative modeling**: Sample and generate new sentences
- **Transfer learning**: Use learned representations for downstream tasks

In [ ]:
# Configuration dictionary# All hyperparameters and settings for the notebook are defined hereCONFIG = {    # General settings    'seed': 42,        # Dataset settings    'dataset_id': 'names',    'train_split': 0.9,    'val_split': 0.1,        # Model architecture (Sequence Autoencoder)    'embedding_dim': 64,    'encoder_hidden': 256,    'decoder_hidden': 256,    'latent_dim': 32,    'num_layers': 2,    'dropout': 0.2,        # Training settings    'batch_size': 128,    'learning_rate': 0.001,    'num_epochs': 1,    'gradient_clip': 1.0,        # Sampling settings    'temperature': 1.0,    'max_length': 20,}print("Configuration loaded:")for key, value in CONFIG.items():    print(f"  {key}: {value}")

## Part 1: Setup and Dependencies

In [ ]:

# Set random seeds for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])

# Device configuration - prioritize MPS (Apple Silicon) if available
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## Part 2: Theory - Sequence-to-Sequence Autoencoders

### Architecture Overview

```
Input Sentence:  "The cat sat on the mat"
       ↓
   ENCODER (RNN/LSTM/GRU)
   Processes tokens sequentially
   Hidden states: h₁, h₂, ..., hₙ
       ↓
   LATENT VECTOR z (fixed size)
   Final hidden state = compressed representation
       ↓
   DECODER (RNN/LSTM/GRU)
   Generates tokens autoregressively
   y₁, y₂, ..., yₘ
       ↓
Output Sentence: "The cat sat on the mat"
```

### Key Concepts

1. **Encoder**: Reads input sequence and produces latent vector z
   - Processes: `x₁, x₂, ..., xₙ → z`
   - Typically uses final hidden state as z

2. **Latent Space**: Fixed-dimensional representation
   - Captures semantic meaning in compact form
   - Similar sentences should have similar latent vectors

3. **Decoder**: Reconstructs sequence from z
   - Generates: `z → y₁, y₂, ..., yₘ`
   - Autoregressive: each token depends on previous tokens

4. **Teacher Forcing**: Training technique
   - Use ground truth token as input (not predicted token)
   - Speeds up training but can cause exposure bias

### Mathematical Formulation

**Encoder**:
```
hₜ = f_enc(xₜ, hₜ₋₁)
z = hₙ  (or pooling of all hidden states)
```

**Decoder**:
```
sₜ = f_dec(yₜ₋₁, sₜ₋₁, z)
yₜ ~ P(y|sₜ)
```

**Loss**: Cross-entropy between predicted and target tokens
```
L = -∑ log P(yₜ | y₁...yₜ₋₁, z)
```

## Part 3: Simple Sentence Dataset

We'll create a small dataset of simple sentences to demonstrate text reconstruction.

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Create a simple sentence dataset
sentences = [
    # Simple subject-verb-object patterns
    "the cat sat on the mat",
    "the dog ran in the park",
    "the bird flew over the tree",
    "the fish swam in the pond",
    "the mouse hid under the chair",
    
    # Location descriptions
    "the book is on the table",
    "the car is in the garage",
    "the phone is on the desk",
    "the key is under the door",
    "the cup is on the shelf",
    
    # Weather and nature
    "the sun shines in the sky",
    "the rain falls from the clouds",
    "the wind blows through the trees",
    "the snow covers the ground",
    "the moon glows at night",
    
    # Actions and activities
    "the child plays with the ball",
    "the teacher writes on the board",
    "the student reads a book",
    "the artist paints a picture",
    "the cook makes a meal",
    
    # Questions and variations
    "where is the cat",
    "what is in the box",
    "who is at the door",
    "when does the train arrive",
    "why is the sky blue",
    
    # More complex patterns
    "the small cat sat quietly",
    "the big dog ran very fast",
    "the blue bird sang sweetly",
    "the old man walked slowly",
    "the young girl smiled happily",
    
    # Compound sentences
    "the cat sat and the dog ran",
    "the bird flew and the fish swam",
    "the sun shines and the wind blows",
    "the child plays and the teacher watches",
    "the rain falls and the ground gets wet",
]

print(f"Dataset size: {len(sentences)} sentences")
print("\nSample sentences:")
for i in range(5):
    print(f"  {i+1}. {sentences[i]}")

### Building a Character-Level Vocabulary

We'll use character-level tokenization for simplicity. This means each character is a token.

In [ ]:
# Build character vocabulary
all_chars = set(''.join(sentences))
all_chars = sorted(list(all_chars))

# Special tokens
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'  # Start of sequence
EOS_TOKEN = '<EOS>'  # End of sequence

# Create vocabulary
vocab = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN] + all_chars
vocab_size = len(vocab)

# Create mappings
char2idx = {ch: idx for idx, ch in enumerate(vocab)}
idx2char = {idx: ch for idx, ch in enumerate(vocab)}

# Constants
PAD_IDX = char2idx[PAD_TOKEN]
SOS_IDX = char2idx[SOS_TOKEN]
EOS_IDX = char2idx[EOS_TOKEN]

print(f"Vocabulary size: {vocab_size}")
print(f"\nSpecial tokens:")
print(f"  PAD: {PAD_IDX}")
print(f"  SOS: {SOS_IDX}")
print(f"  EOS: {EOS_IDX}")
print(f"\nCharacters: {all_chars}")

Display the output.

In [ ]:
# Helper functions for encoding/decoding
def encode_sentence(sentence):
    """Convert sentence to list of character indices."""
    return [char2idx[ch] for ch in sentence]

def decode_sentence(indices):
    """Convert list of indices back to sentence."""
    chars = []
    for idx in indices:
        if idx == EOS_IDX:
            break
        if idx != PAD_IDX and idx != SOS_IDX:
            chars.append(idx2char[idx])
    return ''.join(chars)

# Test encoding/decoding
test_sentence = sentences[0]
encoded = encode_sentence(test_sentence)
decoded = decode_sentence(encoded)

print(f"Original: {test_sentence}")
print(f"Encoded:  {encoded}")
print(f"Decoded:  {decoded}")
print(f"Match: {test_sentence == decoded}")

### Creating a PyTorch Dataset

In [ ]:
class SentenceDataset(Dataset):
    """Dataset for sentence reconstruction."""
    
    def __init__(self, sentences, char2idx, max_len=None):
        self.sentences = sentences
        self.char2idx = char2idx
        
        # Determine max length if not provided
        if max_len is None:
            self.max_len = max(len(s) for s in sentences) + 2  # +2 for SOS/EOS
        else:
            self.max_len = max_len
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        
        # Encode sentence
        encoded = [self.char2idx[ch] for ch in sentence]
        
        # Add SOS and EOS tokens
        input_seq = [SOS_IDX] + encoded  # Input: SOS + sentence
        target_seq = encoded + [EOS_IDX]  # Target: sentence + EOS
        
        # Pad sequences
        input_seq = input_seq + [PAD_IDX] * (self.max_len - len(input_seq))
        target_seq = target_seq + [PAD_IDX] * (self.max_len - len(target_seq))
        
        return {
            'input': torch.tensor(input_seq, dtype=torch.long),
            'target': torch.tensor(target_seq, dtype=torch.long),
            'length': len(encoded) + 1,  # +1 for EOS
            'original': sentence
        }

# Create dataset
dataset = SentenceDataset(sentences, char2idx)

print(f"Dataset size: {len(dataset)}")
print(f"Max sequence length: {dataset.max_len}")

# Examine a sample
sample = dataset[0]
print(f"\nSample:")
print(f"  Original: {sample['original']}")
print(f"  Input shape: {sample['input'].shape}")
print(f"  Target shape: {sample['target'].shape}")
print(f"  Length: {sample['length']}")

Prepare the data loaders for training and validation.

In [ ]:
# Create train/val split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

# Create dataloaders
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"\nBatches per epoch: {len(train_loader)}")

## Part 4: Vanilla Seq2Seq Autoencoder Architecture

We'll implement a simple RNN-based encoder-decoder architecture.

In [ ]:
class Encoder(nn.Module):
    """RNN Encoder that compresses input sequence into latent vector."""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        
        # GRU encoder
        self.gru = nn.GRU(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers,
            batch_first=True
        )
        
        # Project to latent space
        self.hidden_to_latent = nn.Linear(hidden_dim, latent_dim)
    
    def forward(self, x):
        """
        Args:
            x: Input sequence [batch_size, seq_len]
        Returns:
            z: Latent vector [batch_size, latent_dim]
        """
        # Embed input
        embedded = self.embedding(x)  # [batch_size, seq_len, embedding_dim]
        
        # Encode sequence
        _, hidden = self.gru(embedded)  # hidden: [num_layers, batch_size, hidden_dim]
        
        # Use final layer's hidden state
        hidden = hidden[-1]  # [batch_size, hidden_dim]
        
        # Project to latent space
        z = self.hidden_to_latent(hidden)  # [batch_size, latent_dim]
        
        return z

print("Encoder defined successfully")

Define the Decoder architecture.

In [ ]:
class Decoder(nn.Module):
    """RNN Decoder that reconstructs sequence from latent vector."""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        
        # Project latent to initial hidden state
        self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim)
        
        # GRU decoder
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        
        # Output projection
        self.output = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, z):
        """
        Args:
            x: Input tokens [batch_size, seq_len]
            z: Latent vector [batch_size, latent_dim]
        Returns:
            logits: Output logits [batch_size, seq_len, vocab_size]
        """
        batch_size = x.size(0)
        
        # Embed input
        embedded = self.embedding(x)  # [batch_size, seq_len, embedding_dim]
        
        # Initialize hidden state from latent
        hidden = self.latent_to_hidden(z)  # [batch_size, hidden_dim]
        hidden = hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)  # [num_layers, batch_size, hidden_dim]
        
        # Decode sequence
        output, _ = self.gru(embedded, hidden)  # [batch_size, seq_len, hidden_dim]
        
        # Project to vocabulary
        logits = self.output(output)  # [batch_size, seq_len, vocab_size]
        
        return logits

print("Decoder defined successfully")

Define the Seq2SeqAutoencoder architecture.

In [ ]:
class Seq2SeqAutoencoder(nn.Module):
    """Complete sequence-to-sequence autoencoder."""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers=1):
        super().__init__()
        self.encoder = Encoder(vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers)
        self.decoder = Decoder(vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers)
    
    def forward(self, x):
        """
        Args:
            x: Input sequence [batch_size, seq_len]
        Returns:
            logits: Output logits [batch_size, seq_len, vocab_size]
            z: Latent vector [batch_size, latent_dim]
        """
        # Encode
        z = self.encoder(x)
        
        # Decode (teacher forcing - use input as target)
        logits = self.decoder(x, z)
        
        return logits, z
    
    def encode(self, x):
        """Encode input to latent space."""
        return self.encoder(x)
    
    def decode(self, x, z):
        """Decode from latent space."""
        return self.decoder(x, z)
    
    def generate(self, z, max_len=50, temperature=1.0):
        """
        Generate sequence from latent vector (autoregressive).
        
        Args:
            z: Latent vector [batch_size, latent_dim]
            max_len: Maximum generation length
            temperature: Sampling temperature (higher = more random)
        Returns:
            generated: Generated sequences [batch_size, max_len]
        """
        batch_size = z.size(0)
        device = z.device
        
        # Start with SOS token
        generated = torch.full((batch_size, 1), SOS_IDX, dtype=torch.long, device=device)
        
        for _ in range(max_len - 1):
            # Get logits for current sequence
            logits = self.decoder(generated, z)  # [batch_size, current_len, vocab_size]
            
            # Get logits for next token
            next_logits = logits[:, -1, :] / temperature  # [batch_size, vocab_size]
            
            # Sample next token
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)  # [batch_size, 1]
            
            # Append to generated sequence
            generated = torch.cat([generated, next_token], dim=1)
            
            # Check if all sequences have generated EOS
            if (next_token == EOS_IDX).all():
                break
        
        return generated

print("Seq2SeqAutoencoder defined successfully")

Display the output.

In [ ]:
# Initialize model
embedding_dim = 32
hidden_dim = 64
latent_dim = 16
num_layers = 1

model = Seq2SeqAutoencoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    latent_dim=latent_dim,
    num_layers=num_layers
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized on {device}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## Part 5: Training with Teacher Forcing

**Teacher Forcing** is a training technique where:
- We use the **ground truth** token as input at each decoder step
- Instead of feeding the **predicted** token back into the decoder
- This stabilizes training but can cause **exposure bias**

**Without Teacher Forcing** (at inference):
```
Decoder sees: <SOS> → predicts 't' → feeds 't' → predicts 'h' → ...
```

**With Teacher Forcing** (at training):
```
Decoder sees: <SOS> → predicts 't' → feeds 't' (ground truth) → predicts 'h' → ...
```

In [ ]:
# Training configuration
learning_rate=CONFIG['learning_rate']
num_epochs = 100

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(f"Training configuration:")
print(f"  Learning rate: {learning_rate}")
print(f"  Epochs: {num_epochs}")
print(f"  Optimizer: Adam")
print(f"  Loss: CrossEntropyLoss (ignoring padding)")

Evaluate the model on the test set.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for batch in dataloader:
        # Get data
        input_seq = batch['input'].to(device)
        target_seq = batch['target'].to(device)
        
        # Forward pass
        logits, z = model(input_seq)
        
        # Compute loss
        # logits: [batch_size, seq_len, vocab_size]
        # target_seq: [batch_size, seq_len]
        loss = criterion(logits.reshape(-1, vocab_size), target_seq.reshape(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def eval_epoch(model, dataloader, criterion, device):
    """Evaluate for one epoch."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            # Get data
            input_seq = batch['input'].to(device)
            target_seq = batch['target'].to(device)
            
            # Forward pass
            logits, z = model(input_seq)
            
            # Compute loss
            loss = criterion(logits.reshape(-1, vocab_size), target_seq.reshape(-1))
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

print("Training functions defined")

Train the model and monitor progress.

In [ ]:
from tqdm.auto import tqdm

In [ ]:
# Training loop
train_losses = []
val_losses = []
best_val_loss = float('inf')

pbar = tqdm(range(num_epochs), desc="Training")
for epoch in pbar:
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    
    # Validate
    val_loss = eval_epoch(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    
    # Update progress bar
    pbar.set_postfix({
        'train_loss': f'{train_loss:.4f}',
        'val_loss': f'{val_loss:.4f}'
    })
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()

print(f"\nTraining complete!")
print(f"Best validation loss: {best_val_loss:.4f}")

Visualize the results.

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', alpha=0.8)
plt.plot(val_losses, label='Val Loss', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss: {val_losses[-1]:.4f}")

## Part 6: Testing Reconstruction Quality

Let's see how well the model can reconstruct sentences.

In [ ]:
def reconstruct_sentences(model, sentences, char2idx, device, temperature=0.5):
    """Reconstruct sentences using the autoencoder."""
    model.eval()
    reconstructions = []
    
    with torch.no_grad():
        for sentence in sentences:
            # Encode sentence
            encoded = [char2idx[ch] for ch in sentence]
            input_seq = torch.tensor([[SOS_IDX] + encoded], dtype=torch.long, device=device)
            
            # Get latent representation
            z = model.encode(input_seq)
            
            # Generate reconstruction
            generated = model.generate(z, max_len=len(sentence) + 5, temperature=temperature)
            
            # Decode
            reconstruction = decode_sentence(generated[0].cpu().tolist())
            reconstructions.append(reconstruction)
    
    return reconstructions

# Test on validation sentences
test_sentences = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "where is the cat",
    "the sun shines in the sky",
    "the child plays with the ball"
]

reconstructions = reconstruct_sentences(model, test_sentences, char2idx, device, temperature=0.3)

print("Reconstruction Results:\n")
for original, reconstruction in zip(test_sentences, reconstructions):
    match = "✓" if original == reconstruction else "✗"
    print(f"{match} Original:       {original}")
    print(f"  Reconstruction: {reconstruction}")
    print()

### Analyzing Reconstruction Errors

In [ ]:
# Calculate character-level accuracy
def char_accuracy(original, reconstruction):
    """Calculate character-level accuracy."""
    if len(original) == 0:
        return 0.0
    
    matches = sum(1 for a, b in zip(original, reconstruction) if a == b)
    return matches / max(len(original), len(reconstruction))

# Calculate metrics
accuracies = [char_accuracy(orig, recon) for orig, recon in zip(test_sentences, reconstructions)]
exact_matches = sum(1 for orig, recon in zip(test_sentences, reconstructions) if orig == recon)

print(f"Reconstruction Metrics:")
print(f"  Exact matches: {exact_matches}/{len(test_sentences)} ({100*exact_matches/len(test_sentences):.1f}%)")
print(f"  Average char accuracy: {np.mean(accuracies):.3f}")
print(f"  Min char accuracy: {np.min(accuracies):.3f}")
print(f"  Max char accuracy: {np.max(accuracies):.3f}")

## Part 7: Latent Space Visualization with t-SNE

We'll visualize the learned latent representations using t-SNE dimensionality reduction.

In [ ]:
from sklearn.manifold import TSNE

# Encode all sentences
model.eval()
latent_vectors = []
sentence_labels = []

with torch.no_grad():
    for sentence in sentences:
        # Encode
        encoded = [char2idx[ch] for ch in sentence]
        input_seq = torch.tensor([[SOS_IDX] + encoded], dtype=torch.long, device=device)
        z = model.encode(input_seq)
        
        latent_vectors.append(z.cpu().numpy()[0])
        sentence_labels.append(sentence)

latent_vectors = np.array(latent_vectors)

print(f"Latent vectors shape: {latent_vectors.shape}")
print(f"Number of sentences: {len(sentence_labels)}")

Display the output.

In [ ]:
# Apply t-SNE
print("Applying t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(sentences) - 1))
latent_2d = tsne.fit_transform(latent_vectors)

print(f"t-SNE embeddings shape: {latent_2d.shape}")

Visualize the results.

In [ ]:
# Categorize sentences by pattern
def categorize_sentence(sentence):
    """Categorize sentence by pattern."""
    if sentence.startswith("where") or sentence.startswith("what") or \
       sentence.startswith("who") or sentence.startswith("when") or \
       sentence.startswith("why"):
        return "Question"
    elif " and " in sentence:
        return "Compound"
    elif " is " in sentence:
        return "Location"
    elif any(word in sentence for word in ["sun", "rain", "wind", "snow", "moon"]):
        return "Weather"
    else:
        return "Action"

categories = [categorize_sentence(s) for s in sentence_labels]
unique_categories = list(set(categories))
category_colors = plt.cm.tab10(np.linspace(0, 1, len(unique_categories)))
category_to_color = {cat: color for cat, color in zip(unique_categories, category_colors)}

print(f"Categories: {unique_categories}")

Visualize the data distribution with a scatter plot.

In [ ]:
# Plot t-SNE visualization
plt.figure(figsize=(14, 10))

# Plot points colored by category
for category in unique_categories:
    mask = np.array([c == category for c in categories])
    plt.scatter(
        latent_2d[mask, 0],
        latent_2d[mask, 1],
        c=[category_to_color[category]],
        label=category,
        alpha=0.6,
        s=100
    )

# Add sentence labels (sample a few to avoid clutter)
sample_indices = np.random.choice(len(sentences), size=min(10, len(sentences)), replace=False)
for idx in sample_indices:
    plt.annotate(
        sentence_labels[idx][:20] + "...",  # Truncate long sentences
        (latent_2d[idx, 0], latent_2d[idx, 1]),
        fontsize=8,
        alpha=0.7
    )

plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('Latent Space Visualization (t-SNE)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Similar sentences should cluster together")
print("- Different sentence types should form distinct regions")
print("- The latent space captures semantic similarity")

## Part 8: Latent Space Interpolation

One powerful property of autoencoders is that we can **interpolate** between sentences in latent space.

**Interpolation**: Smoothly transitioning from one sentence to another by moving through the latent space.

In [ ]:
def interpolate_sentences(model, sent1, sent2, char2idx, device, num_steps=5, temperature=0.3):
    """
    Interpolate between two sentences in latent space.
    
    Args:
        model: Trained autoencoder
        sent1, sent2: Source and target sentences
        num_steps: Number of interpolation steps
        temperature: Generation temperature
    """
    model.eval()
    
    with torch.no_grad():
        # Encode both sentences
        def encode_sentence(sentence):
            encoded = [char2idx[ch] for ch in sentence]
            input_seq = torch.tensor([[SOS_IDX] + encoded], dtype=torch.long, device=device)
            return model.encode(input_seq)
        
        z1 = encode_sentence(sent1)
        z2 = encode_sentence(sent2)
        
        # Interpolate
        alphas = np.linspace(0, 1, num_steps)
        interpolations = []
        
        for alpha in alphas:
            # Linear interpolation: z = (1-α)z₁ + αz₂
            z_interp = (1 - alpha) * z1 + alpha * z2
            
            # Generate from interpolated latent
            generated = model.generate(z_interp, max_len=50, temperature=temperature)
            reconstruction = decode_sentence(generated[0].cpu().tolist())
            
            interpolations.append((alpha, reconstruction))
    
    return interpolations

print("Interpolation function defined")

Display the output.

In [ ]:
# Test interpolation between different sentence pairs
sentence_pairs = [
    ("the cat sat on the mat", "the dog ran in the park"),
    ("the sun shines in the sky", "the rain falls from the clouds"),
    ("where is the cat", "what is in the box"),
]

for sent1, sent2 in sentence_pairs:
    print(f"\nInterpolating:")
    print(f"  Start: {sent1}")
    print(f"  End:   {sent2}")
    print(f"\nInterpolation steps:")
    
    interpolations = interpolate_sentences(model, sent1, sent2, char2idx, device, num_steps=7)
    
    for alpha, reconstruction in interpolations:
        print(f"  α={alpha:.2f}: {reconstruction}")
    print("-" * 60)

### Reflection: What do you observe?

- Do intermediate sentences make sense?
- Are transitions smooth or abrupt?
- What might improve interpolation quality?

## Part 9: Exploring Random Latent Samples

We can also sample **random points** in the latent space to generate novel sentences.

In [ ]:
# Analyze latent space distribution
latent_mean = latent_vectors.mean(axis=0)
latent_std = latent_vectors.std(axis=0)

print(f"Latent space statistics:")
print(f"  Mean: {latent_mean.mean():.3f} ± {latent_mean.std():.3f}")
print(f"  Std:  {latent_std.mean():.3f} ± {latent_std.std():.3f}")

# Sample random latent vectors
num_samples = 10
random_z = torch.randn(num_samples, latent_dim, device=device)

# Scale to match learned distribution
random_z = random_z * torch.tensor(latent_std.mean(), device=device) + \
           torch.tensor(latent_mean.mean(), device=device)

print(f"\nGenerated {num_samples} random latent vectors")

Evaluate the model on the test set.

In [ ]:
# Generate from random latent vectors
model.eval()
with torch.no_grad():
    generated = model.generate(random_z, max_len=50, temperature=0.5)

print("Random generations from latent space:\n")
for i, gen_seq in enumerate(generated):
    sentence = decode_sentence(gen_seq.cpu().tolist())
    print(f"{i+1:2d}. {sentence}")

## Part 10: Applications and Extensions

### Application 1: Sentence Compression

The latent vector provides a compressed representation of the sentence.

In [ ]:
# Calculate compression ratio
test_sentence = "the cat sat on the mat"

# Original size (characters)
original_size = len(test_sentence)

# Compressed size (latent vector in float32)
compressed_size = latent_dim * 4  # 4 bytes per float32

# Compression ratio
compression_ratio = original_size / compressed_size

print(f"Compression Analysis:")
print(f"  Original sentence: '{test_sentence}'")
print(f"  Original size: {original_size} chars ({original_size} bytes assuming ASCII)")
print(f"  Latent dimension: {latent_dim}")
print(f"  Compressed size: {compressed_size} bytes")
print(f"  Compression ratio: {compression_ratio:.2f}x")
print(f"\nNote: This is lossy compression - reconstruction may not be perfect!")

### Application 2: Sentence Similarity

We can use latent vectors to measure semantic similarity.

In [ ]:
def sentence_similarity(sent1, sent2, model, char2idx, device):
    """Compute cosine similarity between two sentences."""
    model.eval()
    
    with torch.no_grad():
        # Encode sentences
        def encode(sentence):
            encoded = [char2idx[ch] for ch in sentence]
            input_seq = torch.tensor([[SOS_IDX] + encoded], dtype=torch.long, device=device)
            return model.encode(input_seq)
        
        z1 = encode(sent1)
        z2 = encode(sent2)
        
        # Cosine similarity
        similarity = F.cosine_similarity(z1, z2, dim=1).item()
    
    return similarity

# Test similarity
test_pairs = [
    ("the cat sat on the mat", "the cat sat on the mat"),  # Identical
    ("the cat sat on the mat", "the dog sat on the mat"),  # Similar structure
    ("the cat sat on the mat", "the dog ran in the park"), # Different action
    ("the cat sat on the mat", "where is the cat"),        # Different type
    ("the sun shines in the sky", "the rain falls from the clouds"),  # Both weather
]

print("Sentence Similarity Analysis:\n")
for sent1, sent2 in test_pairs:
    sim = sentence_similarity(sent1, sent2, model, char2idx, device)
    print(f"Similarity: {sim:.3f}")
    print(f"  1: {sent1}")
    print(f"  2: {sent2}")
    print()

### Application 3: Analogy and Arithmetic in Latent Space

Similar to word embeddings, we can try **vector arithmetic** in latent space:

```
z_result = z_a - z_b + z_c
```

Example: "cat" - "mat" + "park" = "dog" (in theory)

In [ ]:
def latent_arithmetic(sent_a, sent_b, sent_c, model, char2idx, device, temperature=0.3):
    """
    Perform latent space arithmetic: z_result = z_a - z_b + z_c
    
    Interpretation: "sent_a is to sent_b as sent_c is to ??"
    """
    model.eval()
    
    with torch.no_grad():
        # Encode sentences
        def encode(sentence):
            encoded = [char2idx[ch] for ch in sentence]
            input_seq = torch.tensor([[SOS_IDX] + encoded], dtype=torch.long, device=device)
            return model.encode(input_seq)
        
        z_a = encode(sent_a)
        z_b = encode(sent_b)
        z_c = encode(sent_c)
        
        # Arithmetic
        z_result = z_a - z_b + z_c
        
        # Generate
        generated = model.generate(z_result, max_len=50, temperature=temperature)
        result = decode_sentence(generated[0].cpu().tolist())
    
    return result

# Test analogies
analogies = [
    # Structure: (A, B, C) -> A is to B as C is to ??
    ("the cat sat on the mat", "the dog sat on the mat", "the bird flew over the tree"),
    ("the sun shines in the sky", "the moon glows at night", "the rain falls from the clouds"),
]

print("Latent Space Analogies:\n")
for sent_a, sent_b, sent_c in analogies:
    result = latent_arithmetic(sent_a, sent_b, sent_c, model, char2idx, device)
    print(f"'{sent_a}'")
    print(f"  is to")
    print(f"'{sent_b}'")
    print(f"  as")
    print(f"'{sent_c}'")
    print(f"  is to")
    print(f"'{result}'")
    print("-" * 60)
    print()

## Part 11: Limitations and Improvements

### Current Limitations

1. **Small Dataset**: Only ~35 sentences
2. **Character-level**: Less semantic than word-level
3. **Simple RNN**: GRU is better than vanilla RNN, but still limited
4. **No Attention**: Can't focus on relevant parts of input
5. **Deterministic Latent**: No sampling variability

### Potential Improvements

1. **Larger Dataset**: Train on thousands of sentences
2. **Word-level Tokenization**: Better semantic representations
3. **LSTM/Transformer**: More powerful sequence models
4. **Attention Mechanism**: Focus on relevant input tokens
5. **Variational Autoencoder (VAE)**: Sample from learned distribution
6. **Scheduled Sampling**: Gradually reduce teacher forcing
7. **Beam Search**: Better decoding than greedy/sampling

## Part 12: Extension - Variational Autoencoder (VAE)

A **Variational Autoencoder** improves upon vanilla autoencoders by:
- Learning a **probability distribution** over latent space (not just points)
- Enforcing a **smooth latent space** via KL divergence regularization
- Enabling **sampling** from the learned distribution

### VAE Architecture

```
Encoder → μ (mean), σ (std) → Sample z ~ N(μ, σ) → Decoder
```

### VAE Loss

```
L = Reconstruction Loss + β * KL Divergence
  = -log P(x|z) + β * KL(q(z|x) || p(z))
```

Where:
- **Reconstruction Loss**: How well we reconstruct input
- **KL Divergence**: How close latent distribution is to N(0, 1)
- **β**: Weight balancing reconstruction vs regularization

In [ ]:
class VAEEncoder(nn.Module):
    """Variational Encoder that outputs mean and log-variance."""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        
        # Output mean and log-variance
        self.hidden_to_mu = nn.Linear(hidden_dim, latent_dim)
        self.hidden_to_logvar = nn.Linear(hidden_dim, latent_dim)
    
    def forward(self, x):
        """
        Returns:
            mu: Mean of latent distribution [batch_size, latent_dim]
            logvar: Log-variance of latent distribution [batch_size, latent_dim]
        """
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        hidden = hidden[-1]
        
        mu = self.hidden_to_mu(hidden)
        logvar = self.hidden_to_logvar(hidden)
        
        return mu, logvar

class VariationalSeq2SeqAutoencoder(nn.Module):
    """Variational Seq2Seq Autoencoder."""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers=1):
        super().__init__()
        self.encoder = VAEEncoder(vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers)
        self.decoder = Decoder(vocab_size, embedding_dim, hidden_dim, latent_dim, num_layers)
    
    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick: z = μ + ε * σ, where ε ~ N(0,1)
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode to get distribution parameters
        mu, logvar = self.encoder(x)
        
        # Sample from distribution
        z = self.reparameterize(mu, logvar)
        
        # Decode
        logits = self.decoder(x, z)
        
        return logits, mu, logvar
    
    def generate(self, z, max_len=50, temperature=1.0):
        """Same as regular autoencoder."""
        batch_size = z.size(0)
        device = z.device
        
        generated = torch.full((batch_size, 1), SOS_IDX, dtype=torch.long, device=device)
        
        for _ in range(max_len - 1):
            logits = self.decoder(generated, z)
            next_logits = logits[:, -1, :] / temperature
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            generated = torch.cat([generated, next_token], dim=1)
            
            if (next_token == EOS_IDX).all():
                break
        
        return generated

print("VAE model defined successfully")

Display training results and metrics.

In [ ]:
def vae_loss_function(logits, target, mu, logvar, beta=0.1):
    """
    VAE loss = Reconstruction loss + β * KL divergence
    
    Args:
        logits: Predicted logits [batch_size, seq_len, vocab_size]
        target: Target sequences [batch_size, seq_len]
        mu: Latent mean [batch_size, latent_dim]
        logvar: Latent log-variance [batch_size, latent_dim]
        beta: KL divergence weight
    """
    # Reconstruction loss
    recon_loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target.reshape(-1),
        ignore_index=PAD_IDX
    )
    
    # KL divergence: -0.5 * sum(1 + log(σ²) - μ² - σ²)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    kl_loss = kl_loss / target.size(0)  # Normalize by batch size
    
    # Total loss
    total_loss = recon_loss + beta * kl_loss
    
    return total_loss, recon_loss, kl_loss

print("VAE loss function defined")

### Training the VAE (Optional)

Training a VAE is similar to the vanilla autoencoder, but with the modified loss function.

**Note**: This is left as an exercise. The main differences are:
1. Use `VAEEncoder` instead of `Encoder`
2. Use `vae_loss_function` instead of just cross-entropy
3. Monitor both reconstruction and KL losses
4. Tune the β parameter for good reconstruction vs smooth latent space

## Summary and Key Takeaways

### What We Learned

1. **Seq2Seq Autoencoders**:
   - Encoder compresses variable-length sequences to fixed-size latent vectors
   - Decoder reconstructs sequences from latent representations
   - Useful for unsupervised representation learning

2. **Training Techniques**:
   - Teacher forcing speeds up training
   - Cross-entropy loss for sequence reconstruction
   - Padding and masking for variable-length sequences

3. **Latent Space Properties**:
   - Similar sentences cluster together
   - Can interpolate smoothly between sentences
   - Enables vector arithmetic and analogies

4. **Applications**:
   - Text compression (lossy)
   - Semantic similarity measurement
   - Paraphrase generation
   - Transfer learning (use latent vectors as features)

5. **Extensions**:
   - VAE for probabilistic latent space
   - Attention mechanisms for better context
   - Transformers for state-of-the-art performance

### Reflection Questions

1. Why does teacher forcing cause exposure bias?
2. How would you improve interpolation quality?
3. What are the trade-offs between character-level and word-level models?
4. When would you use a VAE instead of a regular autoencoder?
5. How could you use these latent representations for downstream tasks?

### Next Steps

- Implement VAE and compare to vanilla autoencoder
- Add attention mechanism to encoder-decoder
- Train on larger datasets (e.g., BookCorpus, Wikipedia)
- Experiment with word-level tokenization
- Try Transformer-based autoencoders (e.g., BERT-style models)
- Apply to real-world tasks (sentiment analysis, translation, etc.)